# Weather Dataset — Data Overview

## Objective

This notebook examines the NASA POWER weather dataset used in BharatCrop Intelligence. It establishes the dataset's structure, coverage, parameter definitions, spatial resolution, and limitations before cleaning and preprocessing.

## Key Areas of Inspection

- Dataset dimensions, schema, and data types
- Weather parameters and their units
- Temporal coverage and missing observations
- Grid-point and district-level coverage
- District-to-grid mapping methodology
- Data quality and analytical limitations

## Data Sources

The weather data was collected from NASA POWER for the period 2013–2024, covering precipitation and daily maximum and minimum temperature parameters. Processed data includes annual grid-level summaries and district-level annual records.

## Important Limitations

- Weather summaries are based on calendar years, not crop-specific growing seasons.
- District weather values are representative estimates derived from mapped NASA POWER grid points.
- Spatial and temporal aggregation may hide local weather variations.
- Parameter units and aggregation definitions must be verified before interpreting results.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent

weather_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "district_weather_annual_2013_2024.csv"
)

print("Project root:", project_root)
print("Weather file exists:", weather_path.exists())

if weather_path.exists():
    weather_df = pd.read_csv(weather_path)

    print("\nDataset shape:", weather_df.shape)
    print("\nColumn names:")
    print(weather_df.columns.tolist())

    print("\nFirst five records:")
    display(weather_df.head())
else:
    print("Check the project root and weather dataset path.")

Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Weather file exists: True

Dataset shape: (7680, 24)

Column names:
['apy_geography_id', 'State', 'District', 'district_latitude', 'district_longitude', 'coordinate_source', 'geography_confidence', 'nearest_grid_latitude', 'nearest_grid_longitude', 'nearest_grid_elevation_m', 'nearest_grid_distance_km', 'latitude', 'longitude', 'calendar_year', 'tmin_mean_c', 'tmin_min_c', 'tmin_valid_days', 'tmax_mean_c', 'tmax_max_c', 'tmax_valid_days', 'precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day', 'precipitation_valid_days']

First five records:


,apy_geography_id,State,District,district_latitude,district_longitude,coordinate_source,geography_confidence,nearest_grid_latitude,nearest_grid_longitude,nearest_grid_elevation_m,...,tmin_mean_c,tmin_min_c,tmin_valid_days,tmax_mean_c,tmax_max_c,tmax_valid_days,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,precipitation_valid_days
0,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,27.472904,25.78,365,28.279671,30.11,365,2456.73,6.730767,29.00,365
1,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,27.517479,25.74,365,28.430301,30.75,365,1972.96,5.405370,51.98,365
2,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,27.664959,26.03,365,28.533370,30.15,365,1767.49,4.842438,36.97,365
3,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,27.918333,25.43,366,28.810492,31.98,366,2187.42,5.976557,52.15,366
4,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,27.570219,25.54,365,28.360055,29.62,365,2155.98,5.906795,48.42,365


In [2]:

# Inspect the dataset schema and data types.
schema_summary = pd.DataFrame({
    "Column": weather_df.columns,
    "Data_Type": weather_df.dtypes.astype(str).values,
    "Missing_Count": weather_df.isna().sum().values,
    "Missing_Percent": (
        weather_df.isna().mean().values * 100
    ).round(2),
    "Unique_Values": weather_df.nunique(dropna=True).values,
})

display(schema_summary)

print("Duplicate rows:", weather_df.duplicated().sum())
print("Memory usage (MB):", round(
    weather_df.memory_usage(deep=True).sum() / (1024 ** 2), 2
))


,Column,Data_Type,Missing_Count,Missing_Percent,Unique_Values
0,apy_geography_id,object,0,0.0,640
1,State,object,0,0.0,32
2,District,object,0,0.0,638
3,district_latitude,float64,0,0.0,640
4,district_longitude,float64,0,0.0,640
5,coordinate_source,object,0,0.0,1
6,geography_confidence,object,0,0.0,1
7,nearest_grid_latitude,float64,0,0.0,54
8,nearest_grid_longitude,float64,0,0.0,44
9,nearest_grid_elevation_m,float64,0,0.0,517


Duplicate rows: 0
Memory usage (MB): 3.53


In [3]:

# Inspect temporal and geographic coverage.
print("Year range:", weather_df["calendar_year"].min(),
      "to", weather_df["calendar_year"].max())

print("Number of years:", weather_df["calendar_year"].nunique())
print("Unique districts:", weather_df["apy_geography_id"].nunique())
print("Unique district names:", weather_df["District"].nunique())
print("Unique states:", weather_df["State"].nunique())

print("\nRecords per year:")
display(weather_df.groupby("calendar_year").size().rename("Record_Count"))

print("\nDistrict records per year:")
district_year_counts = weather_df.groupby("apy_geography_id")["calendar_year"].agg(
    Record_Count="count",
    Unique_Years="nunique",
)

display(district_year_counts.describe())

print("\nDistricts not represented in all 12 years:",
      (district_year_counts["Unique_Years"] != 12).sum())

print("Duplicate district-year combinations:",
      weather_df.duplicated(["apy_geography_id", "calendar_year"]).sum())

print("\nRecords by state:")
display(
    weather_df.groupby("State")["apy_geography_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("District_Count")
)


Year range: 2013 to 2024
Number of years: 12
Unique districts: 640
Unique district names: 638
Unique states: 32

Records per year:


calendar_year
2013    640
2014    640
2015    640
2016    640
2017    640
2018    640
2019    640
2020    640
2021    640
2022    640
2023    640
2024    640
Name: Record_Count, dtype: int64


District records per year:


,Record_Count,Unique_Years
count,640.0,640.0
mean,12.0,12.0
std,0.0,0.0
min,12.0,12.0
25%,12.0,12.0
50%,12.0,12.0
75%,12.0,12.0
max,12.0,12.0



Districts not represented in all 12 years: 0
Duplicate district-year combinations: 0

Records by state:


State
Uttar Pradesh                  75
Madhya Pradesh                 53
Rajasthan                      37
Bihar                          37
Tamil Nadu                     36
Maharashtra                    34
Gujarat                        32
Assam                          28
Telangana                      27
Chhattisgarh                   26
Andhra Pradesh                 24
Arunachal Pradesh              23
Odisha                         22
Jharkhand                      22
Haryana                        21
Punjab                         20
Nagaland                       16
Jammu And Kashmir              13
Uttarakhand                    12
Karnataka                      12
Meghalaya                      11
Himachal Pradesh               10
Mizoram                        10
Manipur                         9
West Bengal                     8
Tripura                         7
Sikkim                          6
Andaman And Nicobar Islands     3
Goa                             2
Puducher

In [4]:

# Inspect weather measurement ranges and valid-day counts.

weather_columns = [
    "tmin_mean_c",
    "tmin_min_c",
    "tmax_mean_c",
    "tmax_max_c",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day",
]

print("Weather measurement summary:")
display(weather_df[weather_columns].describe().T)

valid_day_columns = [
    "tmin_valid_days",
    "tmax_valid_days",
    "precipitation_valid_days",
]

print("\nValid-day count summary:")
display(weather_df[valid_day_columns].describe().T)

print("\nDistinct valid-day counts:")
for column in valid_day_columns:
    print(f"{column}: {sorted(weather_df[column].unique())}")

print("\nRecords with fewer than 365 valid days:")
for column in valid_day_columns:
    count = (weather_df[column] < 365).sum()
    print(f"{column}: {count}")


Weather measurement summary:


,count,mean,std,min,25%,50%,75%,max
tmin_mean_c,7680.0,18.903706,4.342466,-11.804208,18.646479,19.664356,20.571610,28.327978
tmin_min_c,7680.0,5.677457,6.490556,-35.460000,3.060000,5.190000,8.660000,26.440000
tmax_mean_c,7680.0,30.412696,4.811445,-0.886831,29.973251,31.852762,32.971226,36.641671
tmax_max_c,7680.0,42.020201,6.189873,14.190000,39.540000,44.100000,46.320000,49.840000
precipitation_total_mm,7680.0,1295.309259,681.737296,136.260000,857.770000,1141.545000,1522.840000,6693.070000
precipitation_mean_mm_day,7680.0,3.546221,1.865906,0.373315,2.349151,3.126740,4.171863,18.337178
precipitation_max_mm_day,7680.0,72.921645,37.143956,13.340000,47.300000,64.715000,88.272500,404.510000



Valid-day count summary:


,count,mean,std,min,25%,50%,75%,max
tmin_valid_days,7680.0,365.25,0.433041,365.0,365.0,365.0,365.25,366.0
tmax_valid_days,7680.0,365.25,0.433041,365.0,365.0,365.0,365.25,366.0
precipitation_valid_days,7680.0,365.25,0.433041,365.0,365.0,365.0,365.25,366.0



Distinct valid-day counts:
tmin_valid_days: [np.int64(365), np.int64(366)]
tmax_valid_days: [np.int64(365), np.int64(366)]
precipitation_valid_days: [np.int64(365), np.int64(366)]

Records with fewer than 365 valid days:
tmin_valid_days: 0
tmax_valid_days: 0
precipitation_valid_days: 0


In [5]:

# Inspect records associated with extreme weather values.

extreme_checks = {
    "Lowest annual mean of daily minimum temperature": (
        "tmin_mean_c", "lowest"
    ),
    "Lowest daily minimum temperature in the year": (
        "tmin_min_c", "lowest"
    ),
    "Highest annual mean of daily maximum temperature": (
        "tmax_mean_c", "highest"
    ),
    "Highest daily maximum temperature in the year": (
        "tmax_max_c", "highest"
    ),
    "Highest annual precipitation total": (
        "precipitation_total_mm", "highest"
    ),
    "Highest daily precipitation": (
        "precipitation_max_mm_day", "highest"
    ),
}

location_columns = [
    "State", "District", "calendar_year",
    "nearest_grid_latitude", "nearest_grid_longitude",
    "nearest_grid_distance_km",
]

for description, (column, direction) in extreme_checks.items():
    print(f"\n{description} — {column}")

    sorted_df = weather_df.sort_values(
        column, ascending=(direction == "lowest")
    )

    display(sorted_df[location_columns + [column]].head(5))



Lowest annual mean of daily minimum temperature — tmin_mean_c


,State,District,calendar_year,nearest_grid_latitude,nearest_grid_longitude,nearest_grid_distance_km,tmin_mean_c
3055,Ladakh,Kargil,2020,34.0,76.25,27.538664,-11.804208
3054,Ladakh,Kargil,2019,34.0,76.25,27.538664,-11.570795
3056,Ladakh,Kargil,2021,34.0,76.25,27.538664,-11.370247
3053,Ladakh,Kargil,2018,34.0,76.25,27.538664,-10.795370
3049,Ladakh,Kargil,2014,34.0,76.25,27.538664,-10.764795



Lowest daily minimum temperature in the year — tmin_min_c


,State,District,calendar_year,nearest_grid_latitude,nearest_grid_longitude,nearest_grid_distance_km,tmin_min_c
3054,Ladakh,Kargil,2019,34.0,76.25,27.538664,-35.46
3055,Ladakh,Kargil,2020,34.0,76.25,27.538664,-33.16
3057,Ladakh,Kargil,2022,34.0,76.25,27.538664,-32.09
3053,Ladakh,Kargil,2018,34.0,76.25,27.538664,-31.36
3059,Ladakh,Kargil,2024,34.0,76.25,27.538664,-31.33



Highest annual mean of daily maximum temperature — tmax_mean_c


,State,District,calendar_year,nearest_grid_latitude,nearest_grid_longitude,nearest_grid_distance_km,tmax_mean_c
2069,Gujarat,Surendranagar,2018,23.0,71.875,32.480483,36.641671
1805,Gujarat,Botad,2018,22.0,71.875,21.864549,36.533260
2009,Gujarat,Patan,2018,24.0,71.875,32.054227,36.349205
1913,Gujarat,Kachchh,2018,23.5,70.000,30.175983,36.241260
2067,Gujarat,Surendranagar,2016,23.0,71.875,32.480483,36.132732


... (2 outputs left out)


Highest annual precipitation total — precipitation_total_mm


,State,District,calendar_year,nearest_grid_latitude,nearest_grid_longitude,nearest_grid_distance_km,precipitation_total_mm
6508,Tripura,South Tripura,2017,23.0,91.875,33.322987,6693.07
4408,Mizoram,Lawngtlai,2017,22.5,92.500,27.883340,5994.72
4420,Mizoram,Lunglei,2017,23.0,92.500,14.716200,5523.71
4327,Meghalaya,West Jaintia Hills,2020,25.5,91.875,31.667409,5363.19
4243,Meghalaya,East Khasi Hills,2020,25.5,91.875,12.252160,5363.19



Highest daily precipitation — precipitation_max_mm_day


,State,District,calendar_year,nearest_grid_latitude,nearest_grid_longitude,nearest_grid_distance_km,precipitation_max_mm_day
2839,Jharkhand,Palamu,2020,24.0,84.375,31.578663,404.51
4283,Meghalaya,South Garo Hills,2024,25.5,90.625,17.567304,389.80
4223,Meghalaya,East Garo Hills,2024,25.5,90.625,13.566989,389.80
7531,Uttarakhand,Pithoragarh,2020,30.0,80.625,19.836278,388.49
2707,Jharkhand,Garhwa,2020,24.0,83.750,11.340615,372.84


In [6]:

# Check internal consistency of annual weather summaries.

consistency_checks = {
    "Mean minimum temperature exceeds mean maximum temperature": (
        weather_df["tmin_mean_c"] > weather_df["tmax_mean_c"]
    ),
    "Annual minimum temperature exceeds annual maximum temperature": (
        weather_df["tmin_min_c"] > weather_df["tmax_max_c"]
    ),
    "Mean minimum temperature is below annual minimum": (
        weather_df["tmin_mean_c"] < weather_df["tmin_min_c"]
    ),
    "Mean maximum temperature exceeds annual maximum": (
        weather_df["tmax_mean_c"] > weather_df["tmax_max_c"]
    ),
    "Precipitation total is below mean daily precipitation": (
        weather_df["precipitation_total_mm"]
        < weather_df["precipitation_mean_mm_day"]
    ),
    "Mean daily precipitation exceeds maximum daily precipitation": (
        weather_df["precipitation_mean_mm_day"]
        > weather_df["precipitation_max_mm_day"]
    ),
}

for description, condition in consistency_checks.items():
    print(f"{description}: {condition.sum()} records")

# Check whether annual precipitation total approximately equals
# mean daily precipitation multiplied by valid days.
expected_precipitation = (
    weather_df["precipitation_mean_mm_day"]
    * weather_df["precipitation_valid_days"]
)

precipitation_difference = (
    weather_df["precipitation_total_mm"] - expected_precipitation
).abs()

print(
    "\nRecords where precipitation total differs from "
    "mean daily precipitation × valid days by more than 0.1 mm:",
    (precipitation_difference > 0.1).sum()
)

print(
    "Maximum absolute precipitation difference (mm):",
    round(precipitation_difference.max(), 4)
)


Mean minimum temperature exceeds mean maximum temperature: 0 records
Annual minimum temperature exceeds annual maximum temperature: 0 records
Mean minimum temperature is below annual minimum: 0 records
Mean maximum temperature exceeds annual maximum: 0 records
Precipitation total is below mean daily precipitation: 0 records
Mean daily precipitation exceeds maximum daily precipitation: 0 records

Records where precipitation total differs from mean daily precipitation × valid days by more than 0.1 mm: 0
Maximum absolute precipitation difference (mm): 0.0


In [7]:

# Assess distance between district representative points and assigned grid points.

distance_summary = weather_df["nearest_grid_distance_km"].describe()

print("District-to-grid distance summary (km):")
display(distance_summary.to_frame("Distance_km"))

distance_thresholds = [10, 25, 50, 100]

print("\nDistricts by distance threshold:")
for threshold in distance_thresholds:
    count = (
        weather_df.loc[
            weather_df["calendar_year"] == weather_df["calendar_year"].min(),
            "nearest_grid_distance_km"
        ] > threshold
    ).sum()

    print(f"More than {threshold} km: {count} districts")

print("\nTen districts with the greatest mapping distances:")
district_mapping = (
    weather_df[
        [
            "apy_geography_id",
            "State",
            "District",
            "district_latitude",
            "district_longitude",
            "nearest_grid_latitude",
            "nearest_grid_longitude",
            "nearest_grid_distance_km",
        ]
    ]
    .drop_duplicates("apy_geography_id")
    .sort_values("nearest_grid_distance_km", ascending=False)
)

display(district_mapping.head(10))


District-to-grid distance summary (km):


,Distance_km
count,7680.000000
mean,23.016620
std,8.572132
min,1.667475
25%,16.849794
50%,23.936196
75%,29.410416
max,41.566652



Districts by distance threshold:
More than 10 km: 585 districts
More than 25 km: 283 districts
More than 50 km: 0 districts
More than 100 km: 0 districts

Ten districts with the greatest mapping distances:


,apy_geography_id,State,District,district_latitude,district_longitude,nearest_grid_latitude,nearest_grid_longitude,nearest_grid_distance_km
3708,APY-GEO-0376,Maharashtra,Akola,20.761445,77.191966,21.0,77.500,41.566652
3180,APY-GEO-0330,Madhya Pradesh,Chhatarpur,24.760161,79.677569,25.0,79.375,40.531506
5868,APY-GEO-0575,Tamil Nadu,Mayiladuthurai,11.191322,79.696424,11.0,80.000,39.368319
5796,APY-GEO-0569,Tamil Nadu,Kallakurichi,11.792456,79.038893,12.0,78.750,38.995600
300,APY-GEO-0027,Andhra Pradesh,Vizianagaram,18.275732,83.466275,18.5,83.750,38.963599
240,APY-GEO-0022,Andhra Pradesh,Sri Potti Sriramulu Nellore,14.717205,79.658459,14.5,79.375,38.904945
1764,APY-GEO-0166,Gujarat,Banas Kantha,24.269692,72.162891,24.5,71.875,38.806143
7152,APY-GEO-0693,Uttar Pradesh,Mainpuri,27.209307,79.062537,27.0,79.375,38.707241
3564,APY-GEO-0364,Madhya Pradesh,Seoni,22.273906,79.720248,22.5,80.000,38.201150
6264,APY-GEO-0615,Telangana,Mulugu,18.313526,80.321813,18.5,80.625,38.120574



## Key Data Overview Findings

### 1. Dataset Structure
- 7,680 records and 24 columns.
- 640 unique district identifiers across 32 states and union territories.
- Annual records cover 2013–2024.

### 2. Temporal Coverage
- Every district has records for all 12 years.
- No duplicate district-year combinations were detected.
- All records contain 365 or 366 valid days for minimum temperature, maximum temperature, and precipitation.

### 3. Missing Values and Internal Consistency
- No missing values were detected in the inspected columns.
- No exact duplicate rows were found.
- Temperature summaries satisfy the tested ordering relationships.
- Annual precipitation totals match mean daily precipitation multiplied by valid days.

### 4. Extreme Values
- The lowest annual mean of daily minimum temperature is approximately -11.80°C in Kargil (2020).
- The highest annual maximum temperature is approximately 49.84°C in Hamirpur (2014).
- The highest annual precipitation total is approximately 6,693.07 mm in South Tripura (2017).
- These observations were retained. Their plausibility should not be treated as proof of correctness; extreme records may require source-level verification.

### 5. Spatial Mapping
- Mean district-to-grid distance: approximately 23.02 km.
- Median distance: approximately 23.94 km.
- Maximum distance: approximately 41.57 km.
- 283 districts have representative points more than 25 km from their assigned grid point.

### 6. Limitations
- Weather summaries use calendar years rather than crop-specific growing seasons.
- District-level values are estimates derived from mapped NASA POWER grid points.
- Complete records and internally consistent summaries do not guarantee source accuracy or local representativeness.
- Weather parameter definitions, units, and aggregation methods should be confirmed before modeling.

### Conclusion

The dataset has complete district-year coverage and passes the internal consistency checks performed in this notebook. It is suitable for proceeding to a dedicated cleaning and validation stage, where suspicious values, spatial mapping limitations, and source definitions will be documented without automatically deleting plausible extremes or imputing missing values.
